<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 06 · E6 — Repeated-run convergence

How many enrichment runs over the *same* glossary does it take for coverage (and
graded score) to plateau? Each run loses some coverage to LLM non-determinism;
repeated runs should recover it. We report the per-run coverage gain and the knee.

> Requires Postgres + `WREN_MEMORY_STORE=none` (see the preconditions cell).
> Onboarding/enrichment make real LLM calls — each round takes a minute or two.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec
import eval_v2 as v2
import seagate_scoring as scoring

FIXTURE = 'seagate_multi'
fdir = v2.fixture_dir(FIXTURE)
manifest = v2.load_table_manifest(FIXTURE)

# Multi-schema project: primary schema seagate_ops + the seagate_core scope.
# seagate_ref is deliberately EXCLUDED (pulling it in is an E9 failure).
config = ec.EvalConfig.from_env(
    schema_name='seagate_ops',
    results_dir=fdir.parent.parent / 'evaluation' / 'results' / 'seagate_multi',
)
client = v2.AgentClientV2(config, schema_names=['seagate_core', 'seagate_ops'])
me = client.login()
print('Authenticated as:', me.get('username') or me)

# Preconditions: Postgres required (R4); memory loop must be off (R1, warn-only).
pre = client.assert_eval_preconditions(require_postgres=True)
print('DB backend:', pre['backend'])
for w in pre['warnings']:
    print('WARNING:', w)

questions = ec.parse_test_queries(fdir / 'test_queries.md')
print('Loaded', len(questions), 'graded questions (Q1-Q18)')
glossary = (fdir / 'bi_glossary.md').read_text(encoding='utf-8')

Authenticated as: admin


DB backend: postgresql
Loaded 18 graded questions (Q1-Q18)


In [2]:
def score_sweep(results):
    """Score a result set with the exact ground-truth scorer (incl. Q16-Q18)."""
    verdicts = {r['id']: scoring.score_result(
        r['id'], r['result_rows'], r['answer_summary']) for r in results}
    correct = sum(1 for v in verdicts.values() if v in scoring.CORRECT_VERDICTS)
    return correct, verdicts

### Clean baseline → resolve → onboard → upload glossary

In [3]:
print('archived', client.clean_baseline(), 'existing project(s)')
project = client.resolve_project()
pid = project['id']
print('project', pid, 'schemas', project.get('schema_names'))
job = client.onboard(pid)
print('onboarded:', (job.get('result') or {}).get('model_count'), 'models')
doc = client.create_document_from_text(pid, glossary, 'bi_glossary.md')
print('document', doc['id'])

archived 1 existing project(s)


project 70de8055-fe96-413e-b53e-45b02d520569 schemas ['seagate_ops', 'seagate_core']


onboarded: 12 models


document 57f84a17-1750-4487-884b-7be6fb1aaded


### Run N enrichment rounds; capture coverage, models, and graded score each round

In [4]:
N = 5
rounds = []
for k in range(1, N + 1):
    signal = client.enrich_round(pid, doc['id'])
    results = ec.run_experiment(client, 'wren_bi', questions, save=False)
    correct, _ = score_sweep(results)
    signal['round'] = k
    signal['graded_correct'] = correct
    signal['n_models'] = len(signal['active_models'])
    rounds.append(signal)
    print(f"round {k}: coverage={signal['coverage']} "
          f"models={signal['n_models']} correct={correct}/18")

round 1: coverage=0.872 models=22 correct=9/18


round 2: coverage=0.872 models=22 correct=8/18


round 3: coverage=0.89 models=22 correct=8/18


round 4: coverage=0.869 models=22 correct=6/18


round 5: coverage=0.869 models=22 correct=7/18


### Convergence curve + marginal gain (the knee)

In [5]:
import pandas as pd
df = pd.DataFrame(rounds)[['round', 'coverage', 'n_models', 'graded_correct']]
df['delta_coverage'] = df['coverage'].diff()
df['delta_correct'] = df['graded_correct'].diff()
print(df.to_string(index=False))
# Knee: first round where marginal coverage gain falls below epsilon.
EPS = 0.02
knee = df[df['delta_coverage'].fillna(1) < EPS]['round'].min()
print('\nRecommended N* (first round with <2% marginal coverage gain):', knee)

 round  coverage  n_models  graded_correct  delta_coverage  delta_correct
     1     0.872        22               9             NaN            NaN
     2     0.872        22               8           0.000           -1.0
     3     0.890        22               8           0.018            0.0
     4     0.869        22               6          -0.021           -2.0
     5     0.869        22               7           0.000            1.0

Recommended N* (first round with <2% marginal coverage gain): 2


**Reading it.** A rising-then-flat coverage curve gives the recommended default
run count `N*`. If `graded_correct` tracks `coverage`, the cheap intrinsic metric is a
valid stand-in (cross-check in notebook 07). Repeat over ≥3 seeds (re-run this
notebook) and average — single runs are noisy (RESULTS.md F9). A round whose
`provenance_kinds` shows no new enrichment event is a converged no-op (R6).